[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/paper_implementation/blob/main/mean_flow_sanity_check.ipynb)

# MeanFlow sanity check — MNIST + ~3.8M DiT, 20k steps

목표는 **검증된 MNIST MeanFlow toy recipe에서 U-Net만 비슷한 체급의 DiT로 바꿨을 때 1-step 생성이 학습되는지** 확인하는 것이다.

기준:
- MNIST \(28\times28\) → pad 2 → \(32\times32\)
- DiT: patch 4, hidden 224, depth 4, heads 8 → 약 3.78M parameters
- batch 128, 20,000 steps
- Adam, lr \(10^{-3}\), betas \((0.9,0.99)\), eps \(10^{-8}\), weight decay 없음
- MeanFlow: logit-normal \((-0.4,1)\), batch 75%에서 \(r=t\), condition \((t,h=t-r)\)
- linked MNIST Colab과 동일하게 adaptive loss의 `norm_eps=1.0`

## 결과 저장 방식

**생성 이미지를 한 장에 합치지 않는다.** `samples/` 폴더에 생성 시점별 PNG를 각각 저장한다.

- `reference_mnist.png`
- `samples/step_00000.png`
- `samples/step_01000.png`
- `samples/step_02000.png`
- ...
- `samples/step_20000.png`

수치 메트릭은 TensorBoard event log에 누적하고, 생성 이미지는 step별 PNG로 각각 저장한다.


## 0. Setup

Colab T4 기준이다. 실행할 때마다 별도 run 폴더를 만든다.


In [ ]:
# @title 0-1. Install / imports / configuration
!pip -q install datasets tensorboard

import math
import os
import random
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from torch.func import jvp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets as tv_datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
from torchvision.utils import save_image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not connected. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then run from the top."
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("CUDA available:", torch.cuda.is_available())

if "T4" not in GPU_NAME:
    warnings.warn(
        f"This notebook is sized for a T4, but current GPU is {GPU_NAME}."
    )

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.mha.set_fastpath_enabled(False)

TRAIN_STEPS = 20_000
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
ADAM_BETAS = (0.9, 0.99)
ADAM_EPS = 1e-8

SCALAR_LOG_EVERY = 50
DIAGNOSTIC_EVERY = 250
SAMPLE_EVERY = 1_000
DIAGNOSTIC_BATCH_SIZE = 64
FIXED_SAMPLE_COUNT = 16
SAMPLE_UPSCALE = 6

P_MEAN = -0.4
P_STD = 1.0
DATA_PROPORTION = 0.75
NORM_P = 1.0
NORM_EPS = 1.0

ROOT_DIR = "/content/meanflow_mnist_dit_sanity"
RUN_NAME = "mnist_dit20k_" + time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(ROOT_DIR, RUN_NAME)
SAMPLE_DIR = os.path.join(RUN_DIR, "samples")
TENSORBOARD_ROOT = os.path.join(ROOT_DIR, "tensorboard")
LOG_DIR = os.path.join(TENSORBOARD_ROOT, RUN_NAME)
REFERENCE_PATH = os.path.join(RUN_DIR, "reference_mnist.png")
CHECKPOINT_PATH = os.path.join(RUN_DIR, "meanflow_dit_mnist_20k.pt")

os.makedirs(SAMPLE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

writer = SummaryWriter(LOG_DIR)
writer.add_text(
    "run/config",
    (
        f"gpu={GPU_NAME}, steps={TRAIN_STEPS}, batch={BATCH_SIZE}, "
        f"lr={LEARNING_RATE}, p_mean={P_MEAN}, p_std={P_STD}, "
        f"data_proportion={DATA_PROPORTION}, norm_eps={NORM_EPS}"
    ),
    global_step=0,
)
writer.flush()

print("RUN_DIR:", RUN_DIR)
print("SAMPLE_DIR:", SAMPLE_DIR)
print("TensorBoard log:", LOG_DIR)


## 1. TensorBoard — 학습 전에 실행

메트릭은 PNG 파일을 계속 덮어쓰지 않고 **TensorBoard event log**에 누적한다.

- `train/loss_adaptive`
- `train/grad_norm`
- `diagnostic/raw_mse_all`
- `diagnostic/raw_mse_interval`
- `diagnostic/interval_cosine`
- `diagnostic/boundary_mse`
- `run/elapsed_minutes`

생성 이미지만 `samples/step_XXXXX.png`로 1000 step마다 각각 별도 저장한다.


In [ ]:
# @title 1-1. Launch TensorBoard before training
%load_ext tensorboard
%tensorboard --logdir /content/meanflow_mnist_dit_sanity/tensorboard --reload_interval 5


## 2. MNIST

Hugging Face의 MNIST를 우선 사용하고, 실패하면 torchvision MNIST로 fallback한다.

입력 변환은 linked MNIST Colab과 같은 의미가 되도록:
1. `ToTensor`
2. pad 2
3. \([0,1]\to[-1,1]\)


In [ ]:
# @title 2-1. Download MNIST and create loaders
to_tensor = ToTensor()


def transform_image(pil_image):
    image = to_tensor(pil_image)
    image = F.pad(
        image,
        (2, 2, 2, 2),
        value=0.0,
    )
    image = image * 2.0 - 1.0
    return image


def hf_collate(batch):
    images = torch.stack(
        [transform_image(item["image"]) for item in batch]
    )
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )
    return images, labels


try:
    hf_dataset = load_dataset("ylecun/mnist")

    train_loader = DataLoader(
        hf_dataset["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
        collate_fn=hf_collate,
    )
    test_loader = DataLoader(
        hf_dataset["test"],
        batch_size=DIAGNOSTIC_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
        collate_fn=hf_collate,
    )
    print("MNIST source: Hugging Face ylecun/mnist")

except Exception as primary_error:
    print("Hugging Face MNIST failed:", repr(primary_error))
    print("Fallback: torchvision.datasets.MNIST")

    fallback_transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Pad(2),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    )

    train_dataset = tv_datasets.MNIST(
        root="/content/mnist_data",
        train=True,
        transform=fallback_transform,
        download=True,
    )
    test_dataset = tv_datasets.MNIST(
        root="/content/mnist_data",
        train=False,
        transform=fallback_transform,
        download=True,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=DIAGNOSTIC_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
    )

train_images, train_labels = next(iter(train_loader))

print("batch shape:", train_images.shape)
print(
    "range:",
    train_images.min().item(),
    train_images.max().item(),
)


## 3. DiT backbone

linked MNIST Colab의 U-Net은 약 3.9M parameters다. 여기서는 약 3.78M의 DiT를 사용한다.

- patch size 4
- hidden dimension 224
- depth 4
- heads 8
- \(t\)와 \(h=t-r\)는 **서로 독립된 embedding MLP**를 사용
- adaLN-Zero 계열의 zero-initialized modulation
- final projection도 zero initialization

`MultiheadAttention(..., need_weights=True)`를 사용해 PyTorch의 SDPA fast path를 피하고 forward-mode JVP가 동작하도록 한다.


In [ ]:
# @title 3-1. ~3.78M MeanFlow DiT
class ScalarEmbed(nn.Module):
    def __init__(self, dim, fourier=64):
        super().__init__()

        self.fourier = fourier
        self.mlp = nn.Sequential(
            nn.Linear(fourier, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, value):
        value = value.reshape(-1, 1)
        half = self.fourier // 2

        frequencies = torch.exp(
            torch.linspace(
                math.log(1.0),
                math.log(1000.0),
                half,
                device=value.device,
            )
        )

        angles = (
            value
            * frequencies[None]
            * 2.0
            * math.pi
        )

        embedding = torch.cat(
            [
                angles.sin(),
                angles.cos(),
            ],
            dim=1,
        )

        return self.mlp(embedding)


class DiTBlock(nn.Module):
    def __init__(self, dim=224, heads=8):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
        )
        self.attention = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
        )

        self.feed_forward = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 6 * dim),
        )

        nn.init.zeros_(self.modulation[-1].weight)
        nn.init.zeros_(self.modulation[-1].bias)

    def forward(self, tokens, condition):
        (
            shift1,
            scale1,
            gate1,
            shift2,
            scale2,
            gate2,
        ) = self.modulation(condition).chunk(
            6,
            dim=-1,
        )

        hidden = (
            self.norm1(tokens)
            * (1.0 + scale1[:, None])
            + shift1[:, None]
        )

        attended, _ = self.attention(
            hidden,
            hidden,
            hidden,
            need_weights=True,
        )

        tokens = (
            tokens
            + gate1[:, None] * attended
        )

        hidden = (
            self.norm2(tokens)
            * (1.0 + scale2[:, None])
            + shift2[:, None]
        )

        tokens = (
            tokens
            + gate2[:, None]
            * self.feed_forward(hidden)
        )

        return tokens


class MeanFlowDiT(nn.Module):
    def __init__(
        self,
        dim=224,
        depth=4,
        heads=8,
        patch=4,
    ):
        super().__init__()

        self.patch = patch
        self.grid = 32 // patch

        self.input_projection = nn.Conv2d(
            1,
            dim,
            kernel_size=patch,
            stride=patch,
        )

        self.position = nn.Parameter(
            torch.randn(
                1,
                self.grid * self.grid,
                dim,
            )
            * 0.02
        )

        self.time_embed = ScalarEmbed(dim)
        self.interval_embed = ScalarEmbed(dim)

        self.blocks = nn.ModuleList(
            [
                DiTBlock(
                    dim=dim,
                    heads=heads,
                )
                for _ in range(depth)
            ]
        )

        self.final_norm = nn.LayerNorm(dim)
        self.final_linear = nn.Linear(
            dim,
            patch * patch,
        )

        nn.init.zeros_(self.final_linear.weight)
        nn.init.zeros_(self.final_linear.bias)

    def forward(self, images, t, h):
        tokens = (
            self.input_projection(images)
            .flatten(2)
            .transpose(1, 2)
            + self.position
        )

        condition = (
            self.time_embed(t)
            + self.interval_embed(h)
        )

        for block in self.blocks:
            tokens = block(
                tokens,
                condition,
            )

        tokens = self.final_norm(tokens)

        patches = self.final_linear(tokens).view(
            images.size(0),
            self.grid,
            self.grid,
            self.patch,
            self.patch,
        )

        images_out = (
            patches.permute(
                0,
                1,
                3,
                2,
                4,
            )
            .reshape(
                images.size(0),
                1,
                32,
                32,
            )
        )

        return images_out


model = MeanFlowDiT().to(DEVICE)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("parameters:", f"{parameter_count:,}")


## 4. MeanFlow objective

경로는

\[
z_t=(1-t)x+t\epsilon,\qquad v=\epsilon-x.
\]

MeanFlow network는

\[
u_\theta(z_t,t,h),\qquad h=t-r
\]

를 출력한다.

두 logit-normal time을 뽑아 큰 값을 \(t\), 작은 값을 \(r\)로 두고, batch의 75%에서는 \(r=t\)로 만든다.

JVP 방향은 linked MNIST Colab과 동일하게

\[
(\dot z_t,\dot t,\dot r)=(v,1,0)
\]

이며 target은

\[
u_{\mathrm{target}}
=
v-(t-r)\frac{d}{dt}u_\theta.
\]

adaptive loss도 linked MNIST Colab과 동일하게 `norm_p=1`, `norm_eps=1.0`을 사용한다.


In [ ]:
# @title 4-1. MeanFlow sampling / JVP / adaptive loss
def sample_logit_normal(
    batch_size,
    device,
):
    normal = torch.randn(
        batch_size,
        device=device,
    )

    return torch.sigmoid(
        normal * P_STD + P_MEAN
    )


def sample_training_tuple(images):
    batch_size = images.size(0)

    time_a = sample_logit_normal(
        batch_size,
        images.device,
    )
    time_b = sample_logit_normal(
        batch_size,
        images.device,
    )

    t = torch.maximum(
        time_a,
        time_b,
    )
    r = torch.minimum(
        time_a,
        time_b,
    )

    boundary_count = int(
        batch_size * DATA_PROPORTION
    )

    r = r.clone()
    r[:boundary_count] = t[:boundary_count]

    noise = torch.randn_like(images)

    t_image = t[
        :,
        None,
        None,
        None,
    ]

    z_t = (
        (1.0 - t_image) * images
        + t_image * noise
    )

    velocity = noise - images

    return (
        z_t,
        velocity,
        r,
        t,
    )


def meanflow_outputs(
    current_model,
    z_t,
    velocity,
    r,
    t,
):
    ones = torch.ones_like(t)
    zeros = torch.zeros_like(r)

    def meanflow_function(
        z_value,
        t_value,
        r_value,
    ):
        interval = t_value - r_value

        return current_model(
            z_value,
            t_value,
            interval,
        )

    prediction, total_derivative = jvp(
        meanflow_function,
        (
            z_t,
            t,
            r,
        ),
        (
            velocity,
            ones,
            zeros,
        ),
    )

    interval_image = (
        t - r
    )[
        :,
        None,
        None,
        None,
    ]

    target = (
        velocity
        - interval_image
        * total_derivative
    ).detach()

    return prediction, target


def meanflow_loss(
    current_model,
    images,
):
    (
        z_t,
        velocity,
        r,
        t,
    ) = sample_training_tuple(images)

    prediction, target = meanflow_outputs(
        current_model,
        z_t,
        velocity,
        r,
        t,
    )

    residual = prediction - target

    per_sample_sse = (
        residual.square()
        .flatten(1)
        .sum(dim=1)
    )

    with torch.no_grad():
        adaptive_weight = 1.0 / (
            per_sample_sse + NORM_EPS
        ).pow(NORM_P)

    loss = (
        per_sample_sse
        * adaptive_weight
    ).mean()

    return loss


## 5. Fixed diagnostics and separate sample files

고정된 test image/noise/time pair로 250 step마다 진단하고 **TensorBoard에 누적**한다.
생성 이미지는 1000 step마다 `samples/step_XXXXX.png`로 각각 독립 저장한다.


In [ ]:
# @title 5-1. Fixed batch / diagnostics / file writers
fixed_generator = torch.Generator().manual_seed(
    SEED + 100
)

fixed_noise = torch.randn(
    FIXED_SAMPLE_COUNT,
    1,
    32,
    32,
    generator=fixed_generator,
).to(DEVICE)

diagnostic_images, _ = next(
    iter(test_loader)
)

diagnostic_images = diagnostic_images[
    :DIAGNOSTIC_BATCH_SIZE
].to(
    DEVICE,
    non_blocking=True,
)

diagnostic_noise = torch.randn(
    diagnostic_images.shape,
    generator=fixed_generator,
).to(DEVICE)

diagnostic_normal_a = torch.randn(
    DIAGNOSTIC_BATCH_SIZE,
    generator=fixed_generator,
)
diagnostic_normal_b = torch.randn(
    DIAGNOSTIC_BATCH_SIZE,
    generator=fixed_generator,
)

diagnostic_time_a = torch.sigmoid(
    diagnostic_normal_a
    * P_STD
    + P_MEAN
).to(DEVICE)

diagnostic_time_b = torch.sigmoid(
    diagnostic_normal_b
    * P_STD
    + P_MEAN
).to(DEVICE)

diagnostic_t = torch.maximum(
    diagnostic_time_a,
    diagnostic_time_b,
)
diagnostic_r = torch.minimum(
    diagnostic_time_a,
    diagnostic_time_b,
)

diagnostic_boundary_count = int(
    DIAGNOSTIC_BATCH_SIZE
    * DATA_PROPORTION
)

diagnostic_r = diagnostic_r.clone()
diagnostic_r[
    :diagnostic_boundary_count
] = diagnostic_t[
    :diagnostic_boundary_count
]

interval_mask = diagnostic_r < diagnostic_t

reference_images = (
    diagnostic_images[
        :FIXED_SAMPLE_COUNT
    ]
    + 1.0
) / 2.0

reference_images = F.interpolate(
    reference_images,
    scale_factor=SAMPLE_UPSCALE,
    mode="nearest",
)

save_image(
    reference_images,
    REFERENCE_PATH,
    nrow=4,
    padding=8,
)

def fixed_diagnostics(current_model):
    t_image = diagnostic_t[
        :,
        None,
        None,
        None,
    ]

    z_t = (
        (1.0 - t_image)
        * diagnostic_images
        + t_image
        * diagnostic_noise
    )

    velocity = (
        diagnostic_noise
        - diagnostic_images
    )

    prediction, target = meanflow_outputs(
        current_model,
        z_t,
        velocity,
        diagnostic_r,
        diagnostic_t,
    )

    residual = prediction - target

    per_sample_mse = (
        residual.square()
        .flatten(1)
        .mean(dim=1)
    )

    cosine_per_sample = F.cosine_similarity(
        prediction.flatten(1),
        target.flatten(1),
        dim=1,
        eps=1e-8,
    )

    boundary_prediction = current_model(
        z_t,
        diagnostic_t,
        torch.zeros_like(
            diagnostic_t
        ),
    )

    boundary_mse = F.mse_loss(
        boundary_prediction,
        velocity,
    )

    return {
        "raw_mse_all": (
            per_sample_mse.mean().item()
        ),
        "raw_mse_interval": (
            per_sample_mse[
                interval_mask
            ]
            .mean()
            .item()
        ),
        "interval_cosine": (
            cosine_per_sample[
                interval_mask
            ]
            .mean()
            .item()
        ),
        "boundary_mse": (
            boundary_mse.item()
        ),
    }


@torch.no_grad()
def generate_one_step(current_model):
    was_training = current_model.training
    current_model.eval()

    ones = torch.ones(
        FIXED_SAMPLE_COUNT,
        device=DEVICE,
    )

    average_velocity = current_model(
        fixed_noise,
        ones,
        ones,
    )

    generated = (
        fixed_noise
        - average_velocity
    )

    if was_training:
        current_model.train()

    return generated


@torch.no_grad()
def save_sample_step(
    current_model,
    step,
):
    generated = generate_one_step(
        current_model
    )

    generated = (
        generated.clamp(-1.0, 1.0)
        + 1.0
    ) / 2.0

    generated = F.interpolate(
        generated,
        scale_factor=SAMPLE_UPSCALE,
        mode="nearest",
    )

    output_path = os.path.join(
        SAMPLE_DIR,
        f"step_{step:05d}.png",
    )

    save_image(
        generated,
        output_path,
        nrow=4,
        padding=8,
    )

    return output_path


print(
    "fixed finite-interval samples:",
    interval_mask.sum().item(),
)
print(
    "reference:",
    REFERENCE_PATH,
)


## 6. Train 20k

- adaptive loss / grad norm은 TensorBoard에 50 step마다 기록
- fixed diagnostics는 TensorBoard에 250 step마다 기록
- 생성은 1000 step마다 **별도 PNG** 저장
- 마지막 checkpoint 저장


In [ ]:
# @title 6-1. Train 20,000 steps
def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=ADAM_BETAS,
    eps=ADAM_EPS,
)

training_iterator = infinite_loader(
    train_loader
)

start_time = time.time()

step_zero_path = save_sample_step(
    model,
    step=0,
)

step_zero_diagnostics = fixed_diagnostics(
    model
)

for metric_name, metric_value in step_zero_diagnostics.items():
    writer.add_scalar(
        f"diagnostic/{metric_name}",
        metric_value,
        0,
    )
writer.add_scalar("train/grad_norm", 0.0, 0)
writer.add_scalar("run/elapsed_minutes", 0.0, 0)
writer.flush()

print(
    "step 0 sample:",
    step_zero_path,
)

model.train()

for step in range(
    1,
    TRAIN_STEPS + 1,
):
    images, _ = next(
        training_iterator
    )

    images = images.to(
        DEVICE,
        non_blocking=True,
    )

    optimizer.zero_grad(
        set_to_none=True,
    )

    loss = meanflow_loss(
        model,
        images,
    )

    loss.backward()

    grad_norm_squared = torch.zeros(
        (),
        device=DEVICE,
    )

    for parameter in model.parameters():
        if parameter.grad is None:
            continue

        grad_norm_squared = (
            grad_norm_squared
            + parameter.grad.detach()
            .square()
            .sum()
        )

    grad_norm = (
        grad_norm_squared
        .sqrt()
        .item()
    )

    optimizer.step()

    if step == 1 or step % SCALAR_LOG_EVERY == 0:
        writer.add_scalar(
            "train/loss_adaptive",
            loss.item(),
            step,
        )
        writer.add_scalar(
            "train/grad_norm",
            grad_norm,
            step,
        )

    if step % DIAGNOSTIC_EVERY == 0:
        diagnostics = fixed_diagnostics(
            model
        )

        elapsed_minutes = (
            time.time() - start_time
        ) / 60.0

        for metric_name, metric_value in diagnostics.items():
            writer.add_scalar(
                f"diagnostic/{metric_name}",
                metric_value,
                step,
            )
        writer.add_scalar(
            "run/elapsed_minutes",
            elapsed_minutes,
            step,
        )
        writer.flush()

        print(
            f"step={step:5d} "
            f"raw_interval="
            f"{diagnostics['raw_mse_interval']:.4f} "
            f"interval_cos="
            f"{diagnostics['interval_cosine']:.4f} "
            f"boundary="
            f"{diagnostics['boundary_mse']:.4f} "
            f"grad="
            f"{grad_norm:.3f} "
            f"minutes="
            f"{elapsed_minutes:.1f}"
        )

    if step % SAMPLE_EVERY == 0:
        sample_path = save_sample_step(
            model,
            step,
        )

        print(
            "saved sample:",
            sample_path,
        )

        model.train()

torch.save(
    {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": TRAIN_STEPS,
        "config": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "betas": ADAM_BETAS,
            "norm_eps": NORM_EPS,
            "p_mean": P_MEAN,
            "p_std": P_STD,
            "data_proportion": DATA_PROPORTION,
        },
    },
    CHECKPOINT_PATH,
)

writer.flush()
writer.close()

print("training complete")
print("checkpoint:", CHECKPOINT_PATH)


## 7. Result paths

생성 결과는 **시점별 독립 PNG**이고, 메트릭은 TensorBoard event log에 있다.


In [ ]:
# @title 7-1. Print result paths
sample_files = sorted(
    file_name
    for file_name in os.listdir(SAMPLE_DIR)
    if file_name.endswith(".png")
)

print("sample files:")
for file_name in sample_files:
    print(" -", os.path.join(SAMPLE_DIR, file_name))

print("TensorBoard root:", TENSORBOARD_ROOT)
print("reference:", REFERENCE_PATH)
print("checkpoint:", CHECKPOINT_PATH)
